# 绳索怎样托住物体？

对应第四、五、六、十三章。所有单元格可以从上到下运行，只需要 NumPy 和 Matplotlib。

我们把物体视为一个点，所有力作用在这个点。重力向下 80 N，两根斜绳沿 3-4-5 三角形的方向向上拉。绳子只能拉不能推；暂不考虑强度、伸长和运动过程。

图中采用英文标注以避免中文字体缺失；正文与问题使用中文。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11, "figure.dpi": 140})
BLUE, ORANGE, GREEN, RED = "#2463a6", "#cb6b17", "#23845b", "#b83b42"
u = np.array([-0.6, 0.8])
v = np.array([0.6, 0.8])
A = np.column_stack([u, v])
gravity = np.array([0.0, -80.0])

def arrow(ax, start, vector, color, label=None):
    start, vector = np.asarray(start), np.asarray(vector)
    ax.annotate("", xy=start + vector, xytext=start,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.5, mutation_scale=16))
    if label:
        ax.plot([], [], color=color, lw=2.5, label=label)

def force_axes(ax, xlim, ylim):
    ax.axhline(0, color="0.6", lw=0.8)
    ax.axvline(0, color="0.6", lw=0.8)
    ax.grid(alpha=0.18)
    ax.set(xlim=xlim, ylim=ylim, aspect="equal",
           xlabel="Horizontal force (N)", ylabel="Vertical force (N)")


## 1. 先预测，再改变两个拉力

先让两根绳子各拉 40 N。你先手算水平、竖直合力，并判断它是否平衡。

运行后把两项改成 50，再改成左 40、右 60。第三次总拉力没变，为什么物体却不平衡？

这里画的是所受的力，不是在模拟物体运动。

In [ ]:
T_left, T_right = 40.0, 40.0  # 改这里：40/40、50/50、40/60
left_force, right_force = T_left * u, T_right * v
rope_sum = left_force + right_force
net_force = rope_sum + gravity
print("两绳合力 (N):", rope_sum)
print("加上重力后的合力 (N):", net_force)
print("是否平衡:", np.allclose(net_force, 0))

fig, ax = plt.subplots(figsize=(6, 6))
for force, color, label in [(left_force, BLUE, "Left rope"),
                            (right_force, ORANGE, "Right rope"),
                            (gravity, RED, "Gravity"),
                            (net_force, GREEN, "Net force")]:
    arrow(ax, [0, 0], force, color, label)
ax.scatter([0], [0], color="0.2", s=25, zorder=5)
force_axes(ax, (-60, 60), (-95, 75))
ax.set_title("Change the tensions; check both components")
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

## 2. 平衡时的力图

左、右绳各拉 50 N。两根斜绳的力都不是竖直的，为什么它们能共同提供竖直的 80 N？指出下图中相互抵消的分量。

箭头长度表示力的大小，不表示实际绳长。

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 6.4))
for force, color, label in [(50*u, BLUE, "Left: (-30, 40) N"),
                            (50*v, ORANGE, "Right: (30, 40) N"),
                            (gravity, RED, "Gravity: (0, -80) N")]:
    arrow(ax, [0, 0], force, color, label)
ax.plot([-30, 30], [40, 40], "--", color="0.7", lw=1)
ax.scatter([0], [0], color="0.2", s=30, zorder=5)
force_axes(ax, (-65, 65), (-95, 75))
ax.set_title("Two rope forces + gravity = zero")
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

## 3. 同一次计算，按行读与按列读

设左、右绳拉力都是 50 N。第一行算水平分量，第二行算竖直分量；按列读，则是两个完整的力相加。

右图把右绳的力平移到左绳力的箭头末端，方向和大小不变。绿色箭头只表示两绳合力，还没有加重力。

In [ ]:
tensions = np.array([50.0, 50.0])
row_result = np.array([-0.6*tensions[0] + 0.6*tensions[1],
                       0.8*tensions[0] + 0.8*tensions[1]])
column_result = tensions[0]*u + tensions[1]*v
assert np.allclose(row_result, column_result)
assert np.allclose(A @ tensions, column_result)

fig, axes = plt.subplots(1, 2, figsize=(10, 5.6))
left_force, right_force = 50*u, 50*v
arrow(axes[0], [0, 0], left_force, BLUE, "Left: 50u")
arrow(axes[0], [0, 0], right_force, ORANGE, "Right: 50v")
arrow(axes[0], [0, 0], gravity, RED, "Gravity")
force_axes(axes[0], (-65, 65), (-95, 95))
axes[0].set_title("Forces at the common point")
axes[0].legend(loc="lower left", fontsize=9)
arrow(axes[1], [0, 0], left_force, BLUE, "50u = (-30, 40)")
arrow(axes[1], left_force, right_force, ORANGE, "50v = (30, 40)")
arrow(axes[1], [0, 0], left_force + right_force, GREEN, "Rope sum = (0, 80)")
force_axes(axes[1], (-65, 65), (-15, 105))
axes[1].set_title("Add the rope forces head to tail")
axes[1].legend(loc="lower left", fontsize=9)
fig.tight_layout()
plt.show()

## 4. 固定绳子方向，改变我们需要的合力

为了考察这两个方向能合成哪些力，我们先把两绳的目标合力写成 (p,q)。目标为 (0,80) 时，它用于抵消原来的重力；若换成其他目标，我们研究的是另一种所需合力，不是说它还能平衡原来的那份重力。

先试目标 (30,80)，手算左、右拉力。再试 (100,0)。后者为什么虽然有实数解，却不能由这两根绳子实现？

In [ ]:
p, q = 30.0, 80.0  # 再试 (100, 0)、(60, 80)、(0, -80)
T_left = q / 1.6 - p / 1.2
T_right = q / 1.6 + p / 1.2
tensions = np.array([T_left, T_right])
target = np.array([p, q])
print("目标两绳合力 (N):", target)
print("所需左、右拉力 (N):", tensions)
print("数学等式成立:", np.allclose(A @ tensions, target))
print("两根绳子都不需要推:", bool(np.all(tensions >= -1e-10)))
assert np.allclose(A @ tensions, target)

# 阴影是非负拉力可达到的区域。实数系数允许的范围则是整个平面。
vertical = np.linspace(0, 120, 200)
fig, ax = plt.subplots(figsize=(6.4, 5.4))
ax.fill_betweenx(vertical, -0.75*vertical, 0.75*vertical,
                 color=GREEN, alpha=0.15, label="Reachable with nonnegative tensions")
ax.plot(-0.75*vertical, vertical, color=BLUE)
ax.plot(0.75*vertical, vertical, color=ORANGE)
ax.scatter([p], [q], color=RED, s=70, zorder=5, label="Chosen target")
force_axes(ax, (-120, 120), (-30, 130))
ax.set_title("Real coefficients vs. physically allowed tensions")
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()

## 5. 增加一根竖直绳，答案还唯一吗？

第三根绳的方向是 (0,1)。先假设它承担 16 N，算出两根斜绳的新拉力；再把 t 改成 80 和 96。

平衡方程只给出可能的分配，不足以决定真实绳子各承担多少力。实际分配还依赖伸长、初始松紧等条件，不能把这里的参数当作实际设备的旋钮。

In [ ]:
t = 16.0  # 再试 0、80、96
A_three = np.column_stack([u, v, [0.0, 1.0]])
x0 = np.array([50.0, 50.0, 0.0])
z = np.array([-0.625, -0.625, 1.0])
chosen = x0 + t*z
print("左、右、竖直绳的拉力 (N):", chosen)
print("两绳/三绳合力 (N):", A_three @ chosen)
print("加上重力 (N):", A_three @ chosen + gravity)
print("非负拉力:", bool(np.all(chosen >= -1e-10)))
assert np.allclose(A_three @ z, 0)
assert np.allclose(A_three @ chosen, -gravity)

parameters = np.linspace(-16, 100, 200)
family = x0[:, None] + z[:, None]*parameters
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.axvspan(0, 80, color=GREEN, alpha=0.12, label="All tensions nonnegative")
# 左、右绳的曲线相同；一条实线、一条虚线，便于识别。
for values, color, style, label in zip(family, [BLUE, ORANGE, GREEN],
                                      ["-", "--", "-"], ["Left", "Right", "Vertical"]):
    ax.plot(parameters, values, color=color, ls=style, lw=2.3, label=label)
ax.axhline(0, color="0.5", lw=0.8)
ax.axvline(t, color=RED, ls=":", label=f"Chosen t = {t:g}")
ax.set(xlabel="Vertical rope tension t (N)", ylabel="Tension (N)",
       title="Different allocations, same resultant (0, 80) N")
ax.grid(alpha=0.2)
ax.legend(ncol=2)
fig.tight_layout()
plt.show()

## 合上实验，自己重建

1. 为什么左绳方向写成 (-0.6,0.8)，而不是 (-3,4)？若使用后者，系数还是拉力大小吗？
2. 从两个方向的平衡，自己推导两根绳子各拉 50 N。
3. 不用求解器，推导目标 (p,q) 的全部拉力公式，并找出非负条件。
4. 为什么增加第三列没有扩大实数可达范围，却让拉力分配不唯一？
5. 分别说明 A、输入和 b 的单位，并解释为什么 b 不是物体的总合力。

物理依据：[OpenStax：静力平衡条件](https://openstax.org/books/university-physics-volume-1/pages/12-1-conditions-for-static-equilibrium)。